In [32]:
%pip show nbformat

Name: nbformat
Version: 5.10.4
Summary: The Jupyter Notebook format
Home-page: https://jupyter.org
Author: 
Author-email: Jupyter Development Team <jupyter@googlegroups.com>
License: BSD 3-Clause License

- Copyright (c) 2001-2015, IPython Development Team
- Copyright (c) 2015-, Jupyter Development Team

All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

1. Redistributions of source code must retain the above copyright notice, this
   list of conditions and the following disclaimer.

2. Redistributions in binary form must reproduce the above copyright notice,
   this list of conditions and the following disclaimer in the documentation
   and/or other materials provided with the distribution.

3. Neither the name of the copyright holder nor the names of its
   contributors may be used to endorse or promote products derived from
   this software without specific prior writte

In [33]:
%pip install --upgrade nbformat ipykernel jupyter plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
from pathlib import Path
import pandas as pd

# 1. Перевірка наявності історичного CSV файлу
file_path = Path("data/official_data_uk.csv")
if not file_path.exists():
    raise FileNotFoundError("DATA MISSING: Файл data/official_data_uk.csv не знайдено. Завантаж архів з Kaggle.")

# 2. Читання даних (low_memory=False для оптимізації RAM)
df = pd.read_csv(file_path, low_memory=False)

# 3. Мапінг колонок
# У цьому датасеті колонки часу вже називаються 'started_at' та 'finished_at'. 
# Потрібно лише перейменувати колонку регіону.
column_mapping = {
    "oblast": "location_title"
}
df = df.rename(columns=column_mapping)

# 4. Context Integrity: Перевірка наявності необхідних ознак
required_cols = ["location_title", "started_at", "finished_at"]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise KeyError(f"DATA MISSING: У файлі відсутні колонки: {missing_cols}. Перевір структуру сирого CSV.")

# 5. Фільтрація масиву для економії пам'яті
# Залишаємо тільки потрібні колонки, відкидаючи деталізацію по районах/громадах (raion, hromada)
df = df[required_cols]

# =====================================================================
# НИЖЧЕ ЗАЛИШАЄШ СВІЙ ПОПЕРЕДНІЙ КОД БЕЗ ЗМІН, ПОЧИНАЮЧИ З ЦЬОГО РЯДКА:
# tz = "Europe/Kyiv"
# def safe_tz_convert(col: pd.Series) -> pd.Series:
# ...
# Далі - наша вже написана логіка:
# 1. df["started_at"] = pd.to_datetime(...)
# 2. Розрахунок тривалості
# 3. Агрегація
# Трансформація часових поясів
tz = "Europe/Kyiv"

def safe_tz_convert(col: pd.Series) -> pd.Series:
    """Парсинг ISO-дат із безпечною конвертацією часових поясів та обробкою NaT."""
    parsed = pd.to_datetime(col, errors="coerce")
    if parsed.empty:
        return parsed
    # Якщо дати парсяться без часового поясу (наприклад, суцільні NaT), локалізуємо в UTC
    if parsed.dt.tz is None:
        parsed = parsed.dt.tz_localize("UTC")
    return parsed.dt.tz_convert(tz)

df["started_at"] = safe_tz_convert(df.get("started_at", pd.Series(dtype='object')))
df["finished_at"] = safe_tz_convert(df.get("finished_at", pd.Series(dtype='object')))

# Логіка активних тривог
df["is_active"] = df["finished_at"].isna()

# Фіксація часу екстракції даних на основі метаданих файлу
file_mtime = pd.Timestamp(file_path.stat().st_mtime, unit="s", tz="UTC").tz_convert(tz)
df["finished_at_calc"] = df["finished_at"].fillna(file_mtime)

# Розрахунок тривалості у хвилинах
df["duration_minutes"] = (df["finished_at_calc"] - df["started_at"]).dt.total_seconds() / 60.0

# Вивід результату
display(df[["location_title", "started_at", "finished_at", "is_active", "duration_minutes"]].head())

,location_title,started_at,finished_at,is_active,duration_minutes
0,Вінницька область,2022-03-15 18:10:34+02:00,2022-03-15 18:50:07+02:00,False,39.550000
1,Житомирська область,2022-03-15 18:11:25+02:00,2022-03-15 18:54:23+02:00,False,42.966667
2,Черкаська область,2022-03-15 18:11:50+02:00,2022-03-15 18:54:47+02:00,False,42.950000
3,Миколаївська область,2022-03-15 18:14:46+02:00,2022-03-15 18:57:08+02:00,False,42.366667
4,Кіровоградська область,2022-03-15 18:15:11+02:00,2022-03-15 18:54:52+02:00,False,39.683333


In [35]:
# Екстракція години доби (формат 24h, від 0 до 23)
df["start_hour"] = df["started_at"].dt.hour

# Екстракція дня тижня.
# Використовуємо pd.Categorical з жорстко заданим порядком, 
# щоб майбутні графіки не сортували дні за алфавітом.
days_order = [
    'Monday', 'Tuesday', 'Wednesday', 
    'Thursday', 'Friday', 'Saturday', 'Sunday'
]
df["start_day"] = pd.Categorical(
    df["started_at"].dt.day_name(), 
    categories=days_order, 
    ordered=True
)

# Створення додаткового бінарного прапорця для вихідних (може бути корисним для EDA)
df["is_weekend"] = df["started_at"].dt.dayofweek >= 5

# Вивід результату для валідації нових ознак
display(df[["started_at", "start_hour", "start_day", "is_weekend"]].head())

,started_at,start_hour,start_day,is_weekend
0,2022-03-15 18:10:34+02:00,18,Tuesday,False
1,2022-03-15 18:11:25+02:00,18,Tuesday,False
2,2022-03-15 18:11:50+02:00,18,Tuesday,False
3,2022-03-15 18:14:46+02:00,18,Tuesday,False
4,2022-03-15 18:15:11+02:00,18,Tuesday,False


In [36]:
# 1. Загальна статистика
total_alerts = len(df)
max_duration_min = df["duration_minutes"].max()
max_duration_hours = max_duration_min / 60.0

# Знаходимо локацію з найдовшою тривогою (idxmax повертає індекс максимального значення)
longest_alert_location = df.loc[df["duration_minutes"].idxmax(), "location_title"]

print("=== ЗАГАЛЬНА СТАТИСТИКА ===")
print(f"Загальна кількість зафіксованих тривог: {total_alerts}")
print(f"Найдовша тривога: {max_duration_hours:.1f} годин(и) ({longest_alert_location})\n")

# 2. Піковий час (Топ-3 години)
print("=== ПІКОВИЙ ЧАС ПОЧАТКУ (ТОП-3 ГОДИНИ) ===")
peak_hours = df["start_hour"].value_counts().head(3)

for hour, count in peak_hours.items():
    # Форматування для зручного читання (наприклад, 02:00)
    print(f"О {hour:02d}:00 -> {count} тривог(и)")
print("\n")

# 3. Топ-5 регіонів за медіанною тривалістю
print("=== ТОП-5 РЕГІОНІВ ЗА МЕДІАННОЮ ТРИВАЛІСТЮ ===")

top_regions = (
    df.groupby("location_title")
    .agg(
        median_duration_min=("duration_minutes", "median"),
        total_alerts=("duration_minutes", "count")
    )
    # Відкидаємо регіони з 1-2 тривогами, щоб статистика була релевантною (опціональний фільтр)
    .query("total_alerts > 0") 
    .sort_values(by="median_duration_min", ascending=False)
    .head(5)
)

# Округлення медіани до цілих чисел для чистоти виводу
top_regions["median_duration_min"] = top_regions["median_duration_min"].round(0)

display(top_regions)

=== ЗАГАЛЬНА СТАТИСТИКА ===
Загальна кількість зафіксованих тривог: 271894
Найдовша тривога: 14497.3 годин(и) (Харківська область)

=== ПІКОВИЙ ЧАС ПОЧАТКУ (ТОП-3 ГОДИНИ) ===
О 21:00 -> 15244 тривог(и)
О 09:00 -> 14212 тривог(и)
О 12:00 -> 13673 тривог(и)


=== ТОП-5 РЕГІОНІВ ЗА МЕДІАННОЮ ТРИВАЛІСТЮ ===


,median_duration_min,total_alerts
location_title,,
Луганська область,5274.0,2
Донецька область,80.0,26647
Чернігівська область,68.0,13103
Сумська область,68.0,18824
Дніпропетровська область,60.0,42069


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# Примусова інтеграція графіків всередині інтерфейсу VS Code
pio.renderers.default = "notebook"

# 1. Підготовка часових компонентів для історичного аналізу
# Переконуємося, що дати мають правильний тип даних
df["started_at"] = pd.to_datetime(df["started_at"])
df["finished_at_calc"] = pd.to_datetime(df["finished_at_calc"])

df["year"] = df["started_at"].dt.year
df["month"] = df["started_at"].dt.month
df["year_month"] = df["started_at"].dt.to_period("M").astype(str)
df["duration_hours"] = df["duration_minutes"] / 60.0

# Мапінг для коректного відображення місяців українською
month_ua = {
    1: "Січ", 2: "Лют", 3: "Бер", 4: "Квіт", 5: "Трав", 6: "Черв",
    7: "Лип", 8: "Серп", 9: "Верес", 10: "Жовт", 11: "Лист", 12: "Груд"
}
df["month_name"] = df["month"].map(month_ua)

# Сортування місяців за календарем, а не за алфавітом
months_order = list(month_ua.values())
df["month_name"] = pd.Categorical(df["month_name"], categories=months_order, ordered=True)


# === ГРАФІК 1: ДИНАМІКА ПО РОКАХ (ЗАГАЛЬНА КІЛЬКІСТЬ) ===
yearly_stats = df.groupby("year").size().reset_index(name="alerts_count")
fig_yearly = px.bar(
    yearly_stats,
    x="year",
    y="alerts_count",
    title="Історична динаміка: Загальна кількість тривог по роках",
    labels={"year": "Рік", "alerts_count": "Кількість тривог"},
    text_auto=True,
    color="alerts_count",
    color_continuous_scale="Blues"
)
fig_yearly.update_layout(xaxis_type="category")
fig_yearly.show()


# === ГРАФІК 2: СЕЗОННІСТЬ ТА ЦИКЛІЧНІСТЬ ПО МІСЯЦЯХ ===
monthly_seasonality = df.groupby("month_name", observed=False).size().reset_index(name="alerts_count")
fig_monthly = px.line(
    monthly_seasonality,
    x="month_name",
    y="alerts_count",
    title="Сезонність: Сумарна кількість тривог за місяцями (акумульовано)",
    labels={"month_name": "Місяць", "alerts_count": "Кількість тривог"},
    markers=True,
    line_shape="spline"
)
fig_monthly.update_traces(line_color="#ef553b", line_width=3)
fig_monthly.show()


# === ГРАФІК 3: ТОП-15 ЛОКАЦІЙ ЗА СУМАРНИМ ЧАСОМ У ТРИВОЗІ ===
top_locations = (
    df.groupby("location_title")["duration_hours"]
    .sum()
    .reset_index(name="total_hours")
    .sort_values(by="total_hours", ascending=False)
    .head(15)
)
fig_top_loc = px.bar(
    top_locations,
    x="total_hours",
    y="location_title",
    orientation="h",
    title="Топ-15 локацій за сумарним часом у стані тривоги (в годинах)",
    labels={"total_hours": "Сумарна тривалість (години)", "location_title": "Регіон/Область"},
    color="total_hours",
    color_continuous_scale="Reds"
)
fig_top_loc.update_layout(yaxis={"categoryorder": "total ascending"})
fig_top_loc.show()


# === ГРАФІК 4: СТРУКТУРА ЧАСУ ДОБИ ПО КАТЕГОРІЯХ ===
def get_day_part(hour: int) -> str:
    if pd.isna(hour): return "Невідомо"
    if 6 <= hour < 12: return "Ранок (06:00-11:59)"
    elif 12 <= hour < 18: return "День (12:00-17:59)"
    elif 18 <= hour < 24: return "Вечір (18:00-23:59)"
    else: return "Ніч (00:00-05:59)"

df["day_part"] = df["start_hour"].apply(get_day_part)
day_part_counts = df["day_part"].value_counts().reset_index(name="count")

fig_pie = px.pie(
    day_part_counts,
    values="count",
    names="day_part",
    hole=0.4,
    title="Розподіл часу початку тривог за періодами доби (Історичні дані)",
    color_discrete_sequence=px.colors.sequential.YlOrRd_r
)
fig_pie.show()


# === ГРАФІК 5: ЕВОЛЮЦІЯ МЕДІАННОЇ ТРИВАЛОСТІ (DEFENSE DATA SCIENCE METRIC) ===
# Розраховуємо медіану для кожного місяця окремо від 2022 року до тепер
duration_evolution = (
    df.groupby("year_month")["duration_minutes"]
    .median()
    .reset_index(name="median_minutes")
    .sort_values("year_month")
)
fig_evolution = px.area(
    duration_evolution,
    x="year_month",
    y="median_minutes",
    title="Еволюція медіанної тривалості однієї тривоги (хв) по місяцях",
    labels={"year_month": "Хронологія (Рік-Місяць)", "median_minutes": "Медіанна тривалість (хвилини)"},
)
fig_evolution.update_traces(line_color="#636EFA", fillcolor="rgba(99, 110, 250, 0.2)")
fig_evolution.update_layout(xaxis_tickangle=-45)
fig_evolution.show()